# 02 — JSON Mode — Practical Examples

**Covers**: OpenAI JSON mode, Anthropic workaround, Gemini JSON mode, type coercion, streaming JSON

In [ ]:
import os, json, re
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. OpenAI JSON Mode — Basic Usage

In [ ]:
def extract_with_json_mode(text: str, schema_hint: str) -> dict:
    """Use OpenAI JSON mode to extract structured data."""
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": f"Extract data as JSON. Schema hint: {schema_hint}"},
            {"role": "user",   "content": text}
        ],
        temperature=0.0
    )
    raw = response.choices[0].message.content
    return json.loads(raw)  # Always safe in JSON mode

# Test it!
result = extract_with_json_mode(
    "John Doe, age 32, email john@example.com, subscribed since 2021.",
    "{name: str, age: int, email: str, member_since: int}"
)
rprint(result)
print(f"Types: {', '.join(f'{k}={type(v).__name__}' for k, v in result.items())}")

## 2. The JSON Mention Rule — What Happens Without It

In [ ]:
# Demonstrate the JSON mention requirement
try:
    bad_response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": "What is 2 + 2?"}]  # No JSON mention!
    )
    # If it somehow works, show output
    print(f"Unexpectedly succeeded: {bad_response.choices[0].message.content}")
except Exception as e:
    print(f"Error (expected): {type(e).__name__}: {e}")

print()
# ✅ With JSON mention
good_response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "Answer arithmetic questions as JSON. Return {result: number}"},
        {"role": "user",   "content": "What is 2 + 2?"}
    ]
)
print(f"Success: {good_response.choices[0].message.content}")

## 3. Type Safety — The Hidden Problem with JSON Mode

In [ ]:
# JSON mode doesn't guarantee types — demonstrate with coercion
def safe_extract_product(text: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "Extract product JSON: {name, price, quantity, available}"},
            {"role": "user",   "content": text}
        ]
    )
    raw_data = json.loads(response.choices[0].message.content)
    print(f"Raw from LLM: {raw_data}")
    print(f"Raw types: {', '.join(f'{k}={type(v).__name__}' for k, v in raw_data.items())}")
    
    # Always coerce types defensively
    return {
        "name":      str(raw_data.get("name", "Unknown")),
        "price":     float(str(raw_data.get("price", 0)).replace("$", "").replace(",", "")),
        "quantity":  int(raw_data.get("quantity", 0)),
        "available": bool(raw_data.get("available", raw_data.get("in_stock", False)))
    }

result = safe_extract_product("Premium Wireless Headphones - $249.99, 15 in stock, currently available")
print(f"\nCoerced result: {result}")
print(f"Coerced types: {', '.join(f'{k}={type(v).__name__}' for k, v in result.items())}")

## 4. JSON Mode with Nested Schema

In [ ]:
# Nested schema via system prompt description
SYSTEM_NESTED = """
Extract order details as JSON with EXACTLY this structure:
{
  "order_id": string,
  "customer": {
    "name": string,
    "email": string
  },
  "items": [
    {"product": string, "quantity": integer, "price": number}
  ],
  "total": number,
  "shipping_method": one of ["standard", "express", "overnight"]
}
"""

order_text = """
Order #B-5521: Customer Maria Garcia (maria@shop.com) ordered:
- 2x Premium Coffee Beans at $14.99 each
- 1x French Press at $34.99
Shipping: Express delivery. Total: $64.97.
"""

response = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": SYSTEM_NESTED},
        {"role": "user",   "content": order_text}
    ],
    temperature=0.0
)

data = json.loads(response.choices[0].message.content)
rprint(data)
print(f"\nCustomer: {data['customer']['name']}")
print(f"Items: {len(data['items'])} items")
print(f"Shipping: {data['shipping_method']}")

## 5. JSON Mode: Anthropic Claude Workaround

In [ ]:
# Anthropic has no native JSON mode — prompt-enforce it + strip fences
# Uncomment and run if you have ANTHROPIC_API_KEY

"""
import anthropic
anthropics_client = anthropic.Anthropic()

def extract_with_claude(text: str, schema_hint: str) -> dict:
    response = anthropic_client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1024,
        system=f\"\"\"You are a data extractor. Always return ONLY valid JSON. No prose. No markdown fences.
        Schema: {schema_hint}\"\"\",
        messages=[{"role": "user", "content": text}]
    )
    raw = response.content[0].text.strip()
    # Strip potential ```json ``` fences
    cleaned = re.sub(r'^```json\s*|\s*```$', '', raw)
    return json.loads(cleaned)

result = extract_with_claude(
    "Tesla Model S, electric car, base price $74,990",
    "{make: str, model: str, type: str, price: number}"
)
print(result)
""";
print("📌 Claude code above — uncomment with your ANTHROPIC_API_KEY")

## 6. JSON Mode: Google Gemini

In [ ]:
# Gemini has native JSON mode via response_mime_type
# Uncomment and run if you have GOOGLE_API_KEY

"""
import google.generativeai as genai
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

gemini_model = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    generation_config=genai.types.GenerationConfig(
        response_mime_type='application/json'  # Gemini JSON mode
    )
)

response = gemini_model.generate_content(
    'Extract {name: str, year: int, genre: str} from: '
    '\"The Shawshank Redemption (1994) is a drama film.\".'
)
data = json.loads(response.text)  # Always valid JSON with Gemini JSON mode
print(data)
""";
print("📌 Gemini code above — uncomment with your GOOGLE_API_KEY")

## 7. Streaming JSON — Large Outputs

In [ ]:
# Stream a large JSON response and parse at the end
stream = client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "Return a JSON object with 5 book recommendations in a 'books' array. Each book has: title, author, year, genre, why_recommended."},
        {"role": "user",   "content": "Recommend 5 books on machine learning."}
    ],
    stream=True
)

full_content = ""
token_count = 0
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        full_content += delta.content
        token_count += 1
        if token_count % 20 == 0:
            print(f"\r  Streaming... {len(full_content)} chars", end="", flush=True)

print(f"\r✅ Received {len(full_content)} chars total")
books_data = json.loads(full_content)
print(f"Number of books: {len(books_data.get('books', []))}")
for book in books_data.get('books', []):
    print(f"  • {book.get('title')} by {book.get('author')} ({book.get('year')})")

## 📌 Key Takeaways

- `response_format={'type': 'json_object'}` → always valid JSON, `json.loads()` safe
- Always include 'JSON' in your prompt or OpenAI raises an error
- Types and field names are NOT guaranteed — always coerce defensively
- Anthropic: prompt-engineer JSON + strip markdown fences
- Gemini: `response_mime_type='application/json'`
- For type safety, upgrade to Strict Structured Outputs (Topic 04)